# Customer Review Analysis - Women Clothing E-Commerce



End-to-End NLP + ML Project

### Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

!pip install wordcloud
from wordcloud import WordCloud

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Load Dataset

In [2]:

df = pd.read_excel('Womens Clothing Reviews Data.xlsx')
df.head()

,Product ID,Category,Subcategory1,SubCategory2,Location,Channel,Customer Age,Review Title,Review Text,Rating,Recommend Flag
0,767,Initmates,Intimate,Intimates,Mumbai,Mobile,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1
1,1080,General,Dresses,Dresses,Bangalore,Mobile,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1
2,1077,General,Dresses,Dresses,Gurgaon,Mobile,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0
3,1049,General Petite,Bottoms,Pants,Chennai,Web,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1
4,847,General,Tops,Blouses,Bangalore,Web,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1


### Data understanding

In [3]:
# Basic info
df.info()

#Check columns
df.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23486 entries, 0 to 23485
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Product ID      23486 non-null  int64 
 1   Category        23472 non-null  object
 2   Subcategory1    23472 non-null  object
 3   SubCategory2    23472 non-null  object
 4   Location        23486 non-null  object
 5   Channel         23486 non-null  object
 6   Customer Age    23486 non-null  int64 
 7   Review Title    19676 non-null  object
 8   Review Text     22641 non-null  object
 9   Rating          23486 non-null  int64 
 10  Recommend Flag  23486 non-null  int64 
dtypes: int64(4), object(7)
memory usage: 2.0+ MB


Index(['Product ID', 'Category', 'Subcategory1', 'SubCategory2', 'Location',
       'Channel', 'Customer Age', 'Review Title', 'Review Text', 'Rating',
       'Recommend Flag'],
      dtype='object')

In [4]:
# CREATE Sentiment Column From Rating
df['sentiment'] = df['Rating'].apply(lambda x: 1 if x >= 3 else 0)

df['sentiment'].value_counts()

sentiment
1    21079
0     2407
Name: count, dtype: int64

### Handle Missing Values

In [5]:
# Check missing values
df.isnull().sum()

Product ID           0
Category            14
Subcategory1        14
SubCategory2        14
Location             0
Channel              0
Customer Age         0
Review Title      3810
Review Text        845
Rating               0
Recommend Flag       0
sentiment            0
dtype: int64

In [6]:
# Fill missing values
df['Review Text'] = df['Review Text'].fillna('')

In [7]:
df['Review Text'].isnull().sum()

0

### Text cleaning and preprocessing

In [30]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

### Feature and Target Selection

In [8]:
X = df['Review Text']
y = df['sentiment']

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

### Feature Engineering using TF-IDF

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=20000,
        ngram_range=(1,2),
        stop_words='english'
    )),
    ('model', LinearSVC(class_weight='balanced'))
])

In [11]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=20000, ngram_range=(1, 2),
                                 stop_words='english')),
                ('model', LinearSVC(class_weight='balanced'))])

### Model Evaluation

In [12]:
from sklearn.metrics import classification_report

y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.49      0.59      0.53       470
           1       0.95      0.93      0.94      4228

    accuracy                           0.90      4698
   macro avg       0.72      0.76      0.74      4698
weighted avg       0.91      0.90      0.90      4698



### Handling Class Imbalance using SMOTE

In [13]:
from imblearn.over_sampling import SMOTE

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2), stop_words='english')

X_tfidf = tfidf.fit_transform(X)

smote = SMOTE(random_state=42)

X_resampled, y_resampled = smote.fit_resample(X_tfidf, y)

### Splitting Data into Train and Test

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled,
    test_size=0.2,
    random_state=42
)

### Model Training (LinearSVC)

In [21]:
from sklearn.svm import LinearSVC

model = LinearSVC()

model.fit(X_train, y_train)

LinearSVC()

### Model Evaluation

In [22]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.90      1.00      0.95      4226
           1       1.00      0.89      0.94      4206

    accuracy                           0.94      8432
   macro avg       0.95      0.94      0.94      8432
weighted avg       0.95      0.94      0.94      8432



### Saving Model and Vectorizer

In [25]:
import pickle

pickle.dump(tfidf, open('tfidf.pkl', 'wb'))
pickle.dump(model, open('model.pkl', 'wb'))

### Testing before deployment

In [31]:
sample1 = "this dress is amazing"
sample2 = "this product is very bad"

clean1 = clean_text(sample1)
clean2 = clean_text(sample2)

vec1 = tfidf.transform([clean1])
vec2 = tfidf.transform([clean2])

print(model.predict(vec1))
print(model.predict(vec2))

[1]
[0]


In [32]:
tests = [
    "very beautiful dress",
    "worst quality ever",
    "not good at all",
    "i love this product",
    "completely ugly"
]

for t in tests:
    clean = clean_text(t)
    vec = tfidf.transform([clean])
    pred = model.predict(vec)[0]
    
    print(t, "->", "Positive" if pred==1 else "Negative")

very beautiful dress -> Positive
worst quality ever -> Negative
not good at all -> Negative
i love this product -> Positive
completely ugly -> Negative
